# Silver Layer — Insurance Charges
**Purpose:** Read raw Bronze data, clean and enrich it into analysis-ready form.
**Input:** `bronze_insurance` (raw, untouched)
**Output:** `silver_insurance` (typed flags + derived categories)
**Why Silver exists:** Bronze = "the truth as it arrived"; Silver = "trustworthy data shaped for analysis."

# Read the Bronze input

In [4]:
# === READ INPUT ===
# Why: Silver's input is the Bronze TABLE, not the web. Each layer reads the previous
# layer's output — that's what makes the pipeline modular and re-runnable.
# Why spark.read.table (not pandas): the data now lives in the lakehouse as a Spark
# Delta table, and all cleaning is real data-processing — Spark's job, not pandas'.
df = spark.read.table("bronze_insurance")

print("Rows read from bronze:", df.count())   # expect 1338
df.printSchema()

StatementMeta(, 4cfa8960-d87d-4725-b1ea-6005b6442a2f, 6, Finished, Available, Finished, False)

Rows read from bronze: 1338
root
 |-- age: long (nullable = true)
 |-- sex: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- children: long (nullable = true)
 |-- smoker: string (nullable = true)
 |-- region: string (nullable = true)
 |-- charges: double (nullable = true)



# import functions

In [7]:
# === IMPORTS ===
# Why: PySpark's transformation tools live in pyspark.sql.functions; we must import
# the specific ones we use.
#   col  -> refer to a column inside an expression
#   when -> if/else builder for creating categories
#   round-> round decimals to a set number of places
from pyspark.sql.functions import col, when, round


StatementMeta(, 4cfa8960-d87d-4725-b1ea-6005b6442a2f, 9, Finished, Available, Finished, False)

## Flag: is_smoker

In [8]:
# === ENRICH: numeric smoker flag ===
# Why we do this: numbers are easier to aggregate than text. A 1/0 flag lets us compute
# "what % are smokers" with a single avg(), and analytics/ML tools prefer numeric flags.
# Why .cast("int") (not when): smoker is BINARY (yes/no), so (col=="yes") already gives
# TRUE/FALSE, and casting that to int yields 1/0. For a two-value flag this is the
# cleanest, most direct expression — no need for when's heavier if/else machinery.
df = df.withColumn("is_smoker", (col("smoker") == "yes").cast("int"))

display(df.select("smoker", "is_smoker").limit(5))

StatementMeta(, 4cfa8960-d87d-4725-b1ea-6005b6442a2f, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 198a04e2-f6a6-4c2f-8ff0-3f5c994a22b9)

## Flag: is_smoker

In [9]:
# === ENRICH: BMI band ===
# Why we do this: "avg charges for obese vs normal" is a far more meaningful dashboard
# slice than raw BMI numbers. Bucketing a continuous value into named groups is a
# core Silver enrichment.
# Why when().when()...otherwise() (not cast): there are FOUR ordered buckets — cast
# can't express that. Spark checks conditions top-to-bottom, FIRST MATCH WINS, so each
# line only needs its UPPER bound (if we got here, the line above already failed, so the
# lower bound is implied). The ORDER is the logic.
df = df.withColumn(
    "bmi_category",
    when(col("bmi") < 18.5, "underweight")
    .when(col("bmi") < 25,  "normal")
    .when(col("bmi") < 30,  "overweight")
    .otherwise("obese")
)

display(df.select("bmi", "bmi_category").limit(10))

StatementMeta(, 4cfa8960-d87d-4725-b1ea-6005b6442a2f, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 80b96c1c-6d7b-47da-935c-3be32e8e8022)

## Category: age_band

In [10]:
# === ENRICH: age band ===
# Why we do this: grouping ages into bands gives cleaner charts and trends than 47
# individual ages. Same first-match-wins pattern as BMI.
# Why when again: multiple ordered ranges -> when is the right tool, and being consistent
# with the BMI logic makes the notebook easy to read.
df = df.withColumn(
    "age_band",
    when(col("age") < 30, "18-29")
    .when(col("age") < 45, "30-44")
    .when(col("age") < 60, "45-59")
    .otherwise("60+")
)

display(df.select("age", "age_band").limit(10))

StatementMeta(, 4cfa8960-d87d-4725-b1ea-6005b6442a2f, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f1f4c21d-1a3d-4953-a767-9c4e1ada09c7)

## Clean: round charges

In [11]:
# === CLEAN: round charges to cents ===
# Why we do this: charges arrive with long tails like 16884.924000003 — noisy for a
# money column. Rounding to 2 decimals makes it read like real currency in reports.
# Why round(col, 2): it's the purpose-built function for fixed decimal places.
# Note: we OVERWRITE 'charges' (same name) — withColumn replaces a column if the name
# already exists. That's intentional; we don't want both a rounded and unrounded version.
df = df.withColumn("charges", round(col("charges"), 2))

display(df.select("charges").limit(5))

StatementMeta(, 4cfa8960-d87d-4725-b1ea-6005b6442a2f, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3e939f9c-43c0-491d-9488-e5f69d05feec)

## Quick sanity check before saving

In [12]:
# === VALIDATE (in memory, before writing) ===
# Why: confirm our new columns look right BEFORE committing to the lakehouse.
# Re-running value_counts-style checks on derived columns proves the logic worked.
print("Smoker flag distribution:")
df.groupBy("is_smoker").count().show()

print("BMI category distribution:")
df.groupBy("bmi_category").count().show()

print("Age band distribution:")
df.groupBy("age_band").count().show()

StatementMeta(, 4cfa8960-d87d-4725-b1ea-6005b6442a2f, 14, Finished, Available, Finished, False)

Smoker flag distribution:
+---------+-----+
|is_smoker|count|
+---------+-----+
|        1|  274|
|        0| 1064|
+---------+-----+

BMI category distribution:
+------------+-----+
|bmi_category|count|
+------------+-----+
|      normal|  225|
|       obese|  707|
|  overweight|  386|
| underweight|   20|
+------------+-----+

Age band distribution:
+--------+-----+
|age_band|count|
+--------+-----+
|     60+|  114|
|   30-44|  392|
|   45-59|  415|
|   18-29|  417|
+--------+-----+



##  Write the Silver table

In [13]:
# === WRITE OUTPUT ===
# Why a separate silver table (not overwrite bronze): we NEVER destroy Bronze — it's our
# replay-able source of truth. Silver is a new, cleaned table alongside it.
# Why overwrite mode: re-running this notebook should refresh silver cleanly, not append
# duplicates.
df.write.mode("overwrite").format("delta").saveAsTable("silver_insurance")

print("Saved 'silver_insurance' to lh_medallion.")

StatementMeta(, 4cfa8960-d87d-4725-b1ea-6005b6442a2f, 15, Finished, Available, Finished, False)

Saved 'silver_insurance' to lh_medallion.


## Verify the saved table

In [19]:
check = spark.read.table("silver_insurance")
print ("Rows in silver:", check.count())
check.printSchema()

StatementMeta(, 4cfa8960-d87d-4725-b1ea-6005b6442a2f, 21, Finished, Available, Finished, False)

Rows in silver: 1338
root
 |-- age: long (nullable = true)
 |-- sex: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- children: long (nullable = true)
 |-- smoker: string (nullable = true)
 |-- region: string (nullable = true)
 |-- charges: double (nullable = true)
 |-- is_smoker: integer (nullable = true)
 |-- bmi_category: string (nullable = true)
 |-- age_band: string (nullable = true)



In [20]:
display(check.limit(5))


StatementMeta(, 4cfa8960-d87d-4725-b1ea-6005b6442a2f, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4ea4a6a0-3d2c-40be-a3ad-28a1b89a1a3b)